In [1]:
from timeit import default_timer
import cantera as ct
import matplotlib.pyplot as plt
import re

input_file = 'john_priori.yaml'
all_species = ct.Species.list_from_file(input_file)
species = []

In [2]:
exclude_string = """CH2*, C2O, HCCOH, CH2OCH, CH2OCH2,
C3H3, pC3H4, aC3H4, cC3H4, aC3H5, CH3CCH2, CH3CHCH, C3H6, nC3H7, iC3H7, C3H8,
CH2CHCO, C2H3CHO, CH3CHOCH2, CH3CH2CHO, CH3COCH3,
C4H2, nC4H3, iC4H3, C4H4, nC4H5, iC4H5, C4H5-2, c-C4H5, C4H6, C4H612, C4H6-2, C4H7, iC4H7,
C4H81, C4H82, iC4H8, pC4H9, sC4H9, iC4H9, tC4H9, C4H10, iC4H10, H2C4O, C4H4O,
CH2CHCHCHO, CH3CHCHCO, C2H3CHOCH2, C4H6O23, CH3CHCHCHO, C4H6O25,
C5H4O, C5H5O(1,3), C5H5O(2,4), C5H4OH, C5H5OH, C5H5, C5H6, lC5H7,
C6H2, C6H3, l-C6H4, o-C6H4, C6H5, C6H6, C6H5CH2, C6H5CH3, C6H5C2H, C6H5O, C6H5OH,
C6H4O2, C6H5CO, C6H5CHO, C6H5CH2OH, OC6H4CH3, HOC6H4CH3, C6H4CH3,
OCCO(401), C2H6N2,
I, CH3I, HI, I2, C2H5I,
C4H10O2, C4H9O, C3H6O3, hv, OCHCHO, HOCO, HOCH(OOH)CHO, HOCH(OO)CHO, OCHCO, HOCHO, OCHO,
CH3ONO, NO, NO2, HNO, N2O, HNO3, cis-HCOH, trans-HCOH, HONO2, HONO, NO3, HNO2,
HCO-exc, HCO-exc2, CH4plusO1Dabs
"""
species_to_exclude = set(s.strip() for s in re.split(r',(?![^(]*\))', exclude_string) if s.strip())
species_to_exclude.discard('')
species_to_exclude.add('CH3NO')   

In [3]:
for S in all_species:
    comp = S.composition
    if 'I' in comp:
        continue

    if S.name in species_to_exclude:
        continue
    
    species.append(S)


species_names = {S.name for S in species}
print('Species: {0}'.format(', '.join(S.name for S in species)))

Species: AR, HE, N2, H, O, OH, HO2, H2, H2O, H2O2, O2, C, CH, CH2, CH3, CH4, HCO, CH2O, CH3O, CH2OH, CH3OH, CO, CO2, C2H, C2H2, H2CC, C2H3, C2H4, C2H5, C2H6, HCCO, CH2CO, CH3CO, CH2CHO, CH3CHO, HOOOOH, O3, O2X, OX, OH*, CH*, HCl, ClOO, ClO, Cl2O2, Cl, Cl2, HOCl


In [4]:
ref_phase = ct.Solution(thermo='ideal-gas', kinetics='gas', species=all_species)
all_reactions = ct.Reaction.list_from_file(input_file, ref_phase)
reactions = []

print('\nReactions:')
for R in all_reactions:
    if not all(reactant in species_names for reactant in R.reactants):
        continue
    if not all(product in species_names for product in R.products):
        continue
    reactions.append(R)
    print(R.equation)
print('\n')

gas1 = ct.Solution(input_file)
gas2 = ct.Solution(name="john_priori_submech3",
                   thermo="ideal-gas", kinetics="gas",
                   transport_model="mixture-averaged",
                   species=species, reactions=reactions)

gas2.update_user_header({"description": "H2/O2 submechanism extracted from john_priori"})
gas2.write_yaml("john_priori_submech3.yaml", header=True)


Reactions:
2 OH (+M) <=> H2O2 (+M)
H2O2 + OH <=> H2O + HO2
2 HO2 <=> H2O2 + O2
2 HO2 => HOOOOH
HOOOOH => 2 HO2
2 HO2 <=> H2O2 + O2X
2 HO2 <=> H2O + O3
2 HO2 <=> O2 + 2 OH
H2O2 + O2X => HOOOOH
HOOOOH => H2O2 + O2X
H2O2 + O2X <=> H2O + O3
H2O2 + O2X <=> O2 + 2 OH
H2O + O3 => HOOOOH
HOOOOH => H2O + O3
H2O + O3 <=> O2 + 2 OH
O2 + 2 OH => HOOOOH
HOOOOH => O2 + 2 OH
HO2 + OH <=> H2O + O2
HO2 + OH <=> H2O + O2X
2 OH <=> H2O + O
CH3 + HO2 <=> CH4 + O2
CH3 + HO2 <=> CH3O + OH
H + HO2 <=> 2 OH
H + HO2 <=> H2O + OX
H + HO2 <=> H2O + O
H + HO2 <=> H2 + O2
CH2O <=> CO + H2
H + HCO <=> CO + H2
CH2O <=> H + HCO
CH2O + O2 <=> HCO + HO2
HCO <=> CO + H
CH4 + OX <=> CH2O + H2
CH4 + OX <=> CH2OH + H
CH4 + OX <=> CH3O + H
CH3OH <=> CH3 + OH
CH3OH <=> CH3O + H
CH3OH <=> CH2OH + H
CH3OH <=> CH2O + H2
CH2OH + H <=> CH3O + H
CH4 + OX <=> CH3OH
CH2O + H2 <=> CH3O + H
CH3 + OH <=> CH3O + H
CH2O + H2 <=> CH2OH + H
CH3 + OH <=> CH2O + H2
CH3 + OH <=> CH2OH + H
CH4 + OX <=> CH3 + OH
2 CH3 <=> C2H5 + H
2 CH3 <=> C2

/tmp/ipykernel_79652/3482547021.py:1: UserWarning: NasaPoly2::validate: 
For species CH2OCH, discontinuity in cp/R detected at Tmid = 500
	Value computed using low-temperature polynomial:  8.393471510000001
	Value computed using high-temperature polynomial: 9.1801039121875

  ref_phase = ct.Solution(thermo='ideal-gas', kinetics='gas', species=all_species)
/tmp/ipykernel_79652/3482547021.py:1: UserWarning: NasaPoly2::validate: 
For species CH2OCH, discontinuity in h/RT detected at Tmid = 500
	Value computed using low-temperature polynomial:  42.199147089791666
	Value computed using high-temperature polynomial: 41.961461604875005

  ref_phase = ct.Solution(thermo='ideal-gas', kinetics='gas', species=all_species)
/tmp/ipykernel_79652/3482547021.py:1: UserWarning: NasaPoly2::validate: 
For species CH2OCH, discontinuity in s/R detected at Tmid = 500
	Value computed using low-temperature polynomial:  33.70692865946735
	Value computed using high-temperature polynomial: 33.51209988778391

  re